# MINUT — Self-contained reproducibility notebook

This notebook reproduces the MINUT **feature extraction**, the **extraction-time benchmark**, and
the **reference-tool** classification benchmark.
The MINUT transform is implemented **inline** below (plain NumPy + optional
Numba), and every path is set in the configuration cell.

Steps:
1. **Configuration & dataset paths** (set two variables).
2. **MINUT transform** — 2D max-intensity binning (self-contained; Numba-accelerated if available).
3. **Reference peak-pickers** (asari, pyOpenMS FeatureFinderMetabo, XCMS) at default parameters.
4. **Downstream classifier + leakage-free evaluation** of the reference tools; MINUT is not scored here — its accuracy is in the `CaseStudy/` notebooks (transform kept only for timing).
5. **Extraction-time table**.
6. **Full-cohort reproduction** (end-to-end orchestration).

**Inputs** (public data):
- COVID-19 plasma: MetaboLights [**MTBLS2291**](https://www.ebi.ac.uk/metabolights/MTBLS2291).
- Lung cancer / tuberculosis serum: MetaboLights [**MTBLS6990**](https://www.ebi.ac.uk/metabolights/MTBLS6990).
- (Milk dataset is private — TOFOO project.)

Each dataset is described by a small CSV *manifest* with columns `sample_name, file_path, label`.

## 0 — Configuration & dataset paths

Edit **`PROJECT_ROOT`** (this repository folder) and **`DATA_DIR`** (folder holding the raw
mzML/mzXML files), or set the `MINUT_DATA_DIR` environment variable.


In [ ]:
from pathlib import Path
import os

# --- Edit these if needed ----------------------------------------------------
PROJECT_ROOT = Path.cwd()                       # repo folder (contains manifests/, scripts/, outputs/)
DATA_DIR = Path(os.environ.get("MINUT_DATA_DIR", str(PROJECT_ROOT / "data")))  # raw mzXML/mzML files
# -----------------------------------------------------------------------------

MANIFEST_DIR = PROJECT_ROOT / "manifests"       # one CSV per dataset: sample_name, file_path, label
OUTPUT_DIR   = PROJECT_ROOT / "outputs"
FEAT_DIR     = OUTPUT_DIR / "features"
LOG_DIR      = OUTPUT_DIR / "logs"
SCRIPTS_DIR  = PROJECT_ROOT / "scripts"         # thin wrappers around the reference peak-pickers
for d in (OUTPUT_DIR, FEAT_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 2024
RSCRIPT = os.environ.get("RSCRIPT", "Rscript")  # path to Rscript for the XCMS baseline
RUN_MODE = "smoke"                              # "smoke" (small subset) or "full" (full cohorts)

MANIFESTS = {ds: MANIFEST_DIR / f"{ds}.csv" for ds in ("covid", "lung", "milk")}
ACCESSIONS = {"covid": "MTBLS2291", "lung": "MTBLS6990", "milk": "private (TOFOO)"}

def _prefer_mzxml(path):
    path = Path(path)
    if path.suffix.lower() == ".mzml":
        alt = path.with_suffix(".mzXML")
        if alt.exists():
            return alt
    return path

def resolve_raw(file_ref):
    p = Path(str(file_ref))
    if p.exists():
        return _prefer_mzxml(p)
    cand = DATA_DIR / p.name
    if cand.exists():
        return _prefer_mzxml(cand)
    # requested file not found as-is: try an .mzXML sibling in DATA_DIR before falling back
    if p.suffix.lower() == ".mzml":
        alt = DATA_DIR / (p.stem + ".mzXML")
        if alt.exists():
            return alt
    return cand if cand.exists() else (DATA_DIR / p)

#print("PROJECT_ROOT :", PROJECT_ROOT)
#print("DATA_DIR     :", DATA_DIR, "  (set env MINUT_DATA_DIR to change)")
#print("OUTPUT_DIR   :", OUTPUT_DIR)
#print("RUN_MODE     :", RUN_MODE)
#print("manifests    :", {k: str(v.relative_to(PROJECT_ROOT)) for k, v in MANIFESTS.items()})

## Environment check

Recommended: Python 3.10+ with `numpy`, `pandas`, `pyteomics`, `scikit-learn`, and (optionally)
`numba`. Numba is not required for correctness — it only accelerates the binning (identical results).


In [16]:
import platform, importlib
print("Python:", platform.python_version())
for m in ["numpy", "pandas", "pyteomics", "sklearn", "numba"]:
    try:
        mod = importlib.import_module(m)
        print(f"  {m:10s} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"  {m:10s} (optional) not found -> {e}" if m == "numba" else f"  {m:10s} MISSING -> {e}")


Python: 3.10.0
  numpy      1.26.4
  pandas     2.3.3
  pyteomics  ?
  sklearn    1.7.2
  numba      0.63.1


## 1 — MINUT transform (self-contained)

MINUT reduces an LC-HRMS run to a fixed 2D grid of `(m/z bin, RT bin)`. For **each bin** it keeps
the single **most intense** ion (max-intensity binning), together with the exact `m/z` and `RT` of
that ion. The output feature vector is the concatenation of three grids flattened row-major:

`[ I (per-bin max intensity) | m/z (normalized) | RT (normalized) ]`

Grid dimensions: `T = floor((RT_max - RT_min)/RT_step)`, `M = floor((mz_max - mz_min)/mz_step)`.

The core loop is JIT-compiled with Numba when available; otherwise it runs as plain Python
(**identical output**, only slower). No external/proprietary code is used.


In [17]:
import numpy as np

# Optional Numba acceleration; falls back to pure Python with identical results.
try:
    from numba import njit
    NUMBA_AVAILABLE = True
except Exception:
    NUMBA_AVAILABLE = False
    def njit(*args, **kwargs):
        if len(args) == 1 and callable(args[0]) and not kwargs:
            return args[0]
        def _wrap(f):
            return f
        return _wrap


@njit(fastmath=True)
def _accumulate_max_bins(rt, mz, inten, start_time, end_time, time_interval,
                         mz_min, mz_max, mz_interval, max_I, max_mz, max_rt):
    """In-place: keep, per (RT bin, m/z bin), the ion of maximum intensity."""
    T = max_I.shape[0]
    M = max_I.shape[1]
    inv_t = 1.0 / time_interval
    inv_m = 1.0 / mz_interval
    for k in range(inten.size):
        r = rt[k]
        if r < start_time or r >= end_time:
            continue
        tb = int((r - start_time) * inv_t)
        if tb < 0 or tb >= T:
            continue
        m = mz[k]
        if m < mz_min or m >= mz_max:
            continue
        mb = int((m - mz_min) * inv_m)
        if mb < 0 or mb >= M:
            continue
        v = inten[k]
        if v > max_I[tb, mb]:
            max_I[tb, mb] = v
            max_mz[tb, mb] = m
            max_rt[tb, mb] = r


def minut_transform(spectra, mz_interval, time_interval, start_time, end_time,
                    mz_min, mz_max, normalize_intensity=True):
    """spectra: list of (rt_minutes, mz_array, intensity_array). Returns the MINUT feature vector."""
    T = int(np.floor((end_time - start_time) / time_interval))
    M = int(np.floor((mz_max - mz_min) / mz_interval))
    max_I = np.zeros((T, M), np.float32)
    max_mz = np.full((T, M), np.nan, np.float32)
    max_rt = np.full((T, M), np.nan, np.float32)
    if spectra:
        rt = np.concatenate([np.full(s[1].size, s[0], np.float32) for s in spectra])
        mz = np.concatenate([np.asarray(s[1], np.float32) for s in spectra])
        it = np.concatenate([np.asarray(s[2], np.float32) for s in spectra])
        _accumulate_max_bins(rt, mz, it, float(start_time), float(end_time), float(time_interval),
                             float(mz_min), float(mz_max), float(mz_interval), max_I, max_mz, max_rt)
    I_flat = max_I.ravel().astype(np.float64)
    if normalize_intensity:
        g = I_flat.max()
        if g > 0:
            I_flat = I_flat / g
    mz_use = np.where(np.isnan(max_mz), mz_min, max_mz)
    rt_use = np.where(np.isnan(max_rt), start_time, max_rt)
    mz_norm = np.clip((mz_use - mz_min) / (mz_max - mz_min), 0, 1).ravel()
    rt_norm = np.clip((rt_use - start_time) / (end_time - start_time), 0, 1).ravel()
    return np.concatenate([I_flat, mz_norm, rt_norm])


from pyteomics import mzml, mzxml

def read_spectra(path):
    """Read MS1 spectra from mzML/mzXML -> list of (rt_minutes, mz_array, intensity_array)."""
    path = str(path)
    low = path.lower()
    out = []
    if low.endswith(".mzml"):
        with mzml.read(path) as reader:
            for s in reader:
                if s.get("ms level") != 1:
                    continue
                sl = s.get("scanList", {})
                scan = (sl.get("scan", [{}]) or [{}])[0] if sl else {}
                rt = scan.get("scan start time", 0.0)
                unit = getattr(rt, "unit_info", None)
                rt = float(rt) / 60.0 if unit == "second" else float(rt)
                out.append((rt, np.asarray(s["m/z array"], np.float32),
                            np.asarray(s["intensity array"], np.float32)))
    elif low.endswith(".mzxml"):
        with mzxml.read(path) as reader:
            for s in reader:
                if s.get("msLevel") != 1:
                    continue
                rt = s.get("retentionTime", 0.0)
                unit = getattr(rt, "unit_info", None)
                rt = float(rt)
                if unit == "second" or (unit is None and rt > 60):
                    rt /= 60.0   # mzXML stores retentionTime in seconds (ISO-8601 PT..S)
                out.append((rt, np.asarray(s["m/z array"], np.float32),
                            np.asarray(s["intensity array"], np.float32)))
    else:
        raise ValueError("unsupported file type: " + path)
    return out


# Bin geometry per dataset (RT in minutes, m/z in Da) — matches the paper exactly.
#   milk : m/z [100,1000) step 10  x RT [2,16) step 2  = 90 x 7  = 630 bins
#   lung : m/z [50,1000)  step 2   x RT [2,12) step 2  = 475 x 5 = 2375 bins
#   covid: m/z [100,700)  step 0.5 x RT [2,20) step 1  = 1200 x 18 = 21600 bins
GEOM = {
    "covid": dict(mz_min=100., mz_max=700.,  mz_interval=0.5, start_time=2., end_time=20., time_interval=1.),
    "lung":  dict(mz_min=50.,  mz_max=1000., mz_interval=2.0, start_time=2., end_time=12., time_interval=2.),
    "milk":  dict(mz_min=100., mz_max=1000., mz_interval=10., start_time=2., end_time=16., time_interval=2.),
}

def grid_dims(g):
    return (int(np.floor((g["end_time"] - g["start_time"]) / g["time_interval"])),
            int(np.floor((g["mz_max"] - g["mz_min"]) / g["mz_interval"])))

print("NUMBA_AVAILABLE:", NUMBA_AVAILABLE)
print("bins per dataset:", {k: grid_dims(v)[0] * grid_dims(v)[1] for k, v in GEOM.items()})


NUMBA_AVAILABLE: True
bins per dataset: {'covid': 21600, 'lung': 2375, 'milk': 630}


### 1.a — Extract MINUT features for one dataset (with timing)

Reads each file listed in the manifest, computes the MINUT feature vector, writes
`outputs/features/<dataset>_minut.csv`, and reports the total wall-clock time.


In [18]:
import time
import pandas as pd

TIMINGS = {}   # (dataset, tool) -> minutes

def extract_minut(dataset, manifest, limit=None, out_name=None):
    g = GEOM[dataset if dataset in GEOM else dataset.split("_")[0]]
    T, M = grid_dims(g); n_bins = T * M
    man = pd.read_csv(manifest)
    if limit:
        man = man.head(limit)
    col = "file_path" if "file_path" in man.columns else man.columns[0]
    print(f"[MINUT] {dataset}: {len(man)} files, {n_bins} bins (T={T}, M={M}), numba={NUMBA_AVAILABLE}")
    feats = np.zeros((len(man), n_bins), np.float32)
    t0 = time.time()
    for i, row in man.reset_index(drop=True).iterrows():
        t1 = time.time()
        try:
            spectra = read_spectra(resolve_raw(row[col]))
            v = minut_transform(spectra, g["mz_interval"], g["time_interval"],
                                g["start_time"], g["end_time"], g["mz_min"], g["mz_max"])
            feats[i, :] = v[:n_bins]   # keep the intensity block as the feature matrix
        except Exception as e:
            print(f"  [{i+1:>3}] FAIL {row.get('sample_name', i)}: {e}")
            continue
        if i == 0 or (i + 1) % 10 == 0:
            print(f"  [{i+1:>3}/{len(man)}] {time.time()-t1:5.2f}s", flush=True)
    elapsed = (time.time() - t0) / 60
    df = pd.DataFrame(feats, columns=[f"bin_{i}" for i in range(n_bins)])
    df.insert(0, "sample_name", man["sample_name"].values if "sample_name" in man.columns else np.arange(len(man)))
    if "label" in man.columns:
        df["label"] = man["label"].values
    out = FEAT_DIR / (out_name or f"{dataset}_minut.csv")
    df.to_csv(out, index=False)
    TIMINGS[(dataset, "minut")] = elapsed
    print(f"[MINUT] {dataset} -> {out.name}   Total elapsed: {elapsed:.2f} min")
    return out, elapsed


In [19]:
# Quick end-to-end test (smoke mode): MINUT on a few files from the milk manifest.
if RUN_MODE == "smoke":
    man = MANIFESTS["milk"]
    if Path(man).exists():
        extract_minut("milk", man, limit=3, out_name="milk_minut_SMOKE.csv")
    else:
        print("manifest not found:", man, "-> create manifests/milk.csv (sample_name, file_path, label)")
else:
    print("RUN_MODE='full': use run_all(...) in section 6.")


IndentationError: expected an indented block after 'else' statement on line 6 (1238765407.py, line 8)

## 2 — Reference peak-pickers (default parameters)

For a fair, tuning-free comparison we run three established tools at their **published defaults**:
asari, pyOpenMS **FeatureFinderMetabo** (FFM), and **XCMS**. Each is a small, self-explanatory
wrapper (provided in `scripts/`) that reads a manifest and writes a `samples x features` matrix to
`outputs/features/<dataset>_<tool>.csv`. Interfaces:

| Tool | Wrapper | Invocation |
|------|---------|-----------|
| FFM (pyOpenMS) | `scripts/extract_ffm.py`   | `python scripts/extract_ffm.py --dataset <ds> --manifest <csv>` |
| asari          | `scripts/extract_asari.py` | `python scripts/extract_asari.py --dataset <ds> --manifest <csv>` |
| XCMS (R)       | `scripts/extract_xcms.R`   | `Rscript scripts/extract_xcms.R <ds> <csv>` |

> On full cohorts these tools can take **tens of minutes to hours**; the MINUT transform above
> stays on the order of a minute.


In [ ]:
import subprocess, sys

def run_reference_tool(tool, dataset, manifest, timeout=None):
    """Run a reference peak-picker wrapper (subprocess), log and time it (minutes)."""
    manifest = str(manifest)
    log = LOG_DIR / f"{tool}_{dataset}.log"
    if tool == "xcms":
        cmd = [RSCRIPT, str(SCRIPTS_DIR / "extract_xcms.R"), dataset, manifest]
    else:
        cmd = [sys.executable, str(SCRIPTS_DIR / f"extract_{tool}.py"), "--dataset", dataset, "--manifest", manifest]
    print(f"[RUN] {dataset}/{tool}: {' '.join(cmd)}", flush=True)
    t0 = time.time()
    env = dict(os.environ, PYTHONIOENCODING="utf-8",
               MINUT_FEATURES_DIR=str(FEAT_DIR), MINUT_DATA_DIR=str(DATA_DIR))
    with open(log, "w", encoding="utf-8") as fh:
        cp = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env, timeout=timeout)
    dt = (time.time() - t0) / 60
    out = FEAT_DIR / f"{dataset}_{tool}.csv"
    ok = cp.returncode == 0 and out.exists()
    TIMINGS[(dataset, tool)] = dt
    print(f"[{'OK' if ok else 'FAIL'}] {dataset}/{tool}  {dt:.2f} min  rc={cp.returncode}")
    return ok, dt


In [ ]:
# Example (uncomment to run a reference tool):
# run_reference_tool("ffm",   "covid", MANIFESTS["covid"])
# run_reference_tool("asari", "milk",  MANIFESTS["milk"])
# run_reference_tool("xcms",  "lung",  MANIFESTS["lung"])
print("Uncomment run_reference_tool(...) to run the baselines.")


Uncomment run_reference_tool(...) to run the baselines.


## 3 — Downstream classifier & leakage-free evaluation (reference tools only)

Each **reference-tool** feature matrix (asari, FFM, XCMS) is fed to the **same** classifier and
protocol, matching the article benchmark:
- `RandomForestClassifier(n_estimators=100, random_state=2024)`;
- **10-fold CV** (accuracy, AUC: mean +/- std), reported as the training cross-validation score;
- **hold-out** 75/25 split (`random_state=2024`) for datasets with one; the **COVID** cohort
  (`n=63`) is reported with **cross-validation only** — no predefined hold-out split exists;
- `StandardScaler(with_mean=False)`, NaN->0; scaling refit inside each fold (no leakage).

**MINUT is intentionally not scored here** — its transform is kept only for the timing benchmark
(section 4), and its accuracy / biomarker analysis is performed in the `CaseStudy/` notebooks,
which apply the paper's in-fold Gini-importance and Mann–Whitney + Fisher selection.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.pipeline import Pipeline

def _rf():
    return Pipeline([("sc", StandardScaler(with_mean=False)),
                     ("rf", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED))])

def evaluate_features(path, holdout=True):
    """Benchmark evaluation for the reference tools: RF(100 trees, random_state=2024),
    10-fold cross-validation (mean +/- std). `holdout=False` for cohorts without a
    predefined test split (COVID, n=63), which are reported with CV only."""
    df = pd.read_csv(path, low_memory=False)
    if "label" not in df.columns:
        raise ValueError("no 'label' column in " + str(path))
    y = df["label"].to_numpy().astype(int)
    X = df.drop(columns=["sample_name", "label"], errors="ignore").to_numpy(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    accs, aucs = [], []
    for tr, te in KFold(10, shuffle=True, random_state=SEED).split(X, y):
        pipe = _rf().fit(X[tr], y[tr]); p = pipe.predict_proba(X[te])[:, 1]
        accs.append(accuracy_score(y[te], (p > 0.5).astype(int)))
        if len(np.unique(y[te])) > 1:                # AUC is undefined on a single-class fold
            aucs.append(roc_auc_score(y[te], p))
    out = dict(n_samples=X.shape[0], n_features=X.shape[1],
               cv_acc=float(np.mean(accs)), cv_acc_std=float(np.std(accs)),
               cv_auc=float(np.mean(aucs)) if aucs else float("nan"),
               cv_auc_std=float(np.std(aucs)) if aucs else float("nan"),
               ho_acc=None, ho_auc=None)
    if holdout:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=SEED)
        pipe = _rf().fit(Xtr, ytr); p = pipe.predict_proba(Xte)[:, 1]
        out["ho_acc"] = float(accuracy_score(yte, (p > 0.5).astype(int)))
        out["ho_auc"] = float(roc_auc_score(yte, p))
    return out

In [ ]:
# Evaluate every reference-tool feature matrix in outputs/features/.
# MINUT is intentionally NOT scored here: its transform is kept only for the timing benchmark,
# and its accuracy / biomarker analysis lives in the CaseStudy/ notebooks.
rows = []
for csv_path in sorted(FEAT_DIR.glob("*.csv")):
    name = csv_path.stem
    if "_" not in name:
        continue
    ds, tool = name.rsplit("_", 1)
    if tool == "minut":            # skip MINUT: timed only, accuracy is in the case studies
        continue
    try:
        m = evaluate_features(csv_path, holdout=(ds != "covid"))   # COVID (n=63): CV only
        m.update(dataset=ds, tool=tool); rows.append(m)
        ho = f"{m['ho_auc']:.3f}" if m['ho_auc'] is not None else "  -- "
        print(f"  {ds:14s}/{tool:6s}  n={m['n_samples']:>4} feats={m['n_features']:>6}  "
              f"CV-AUC={m['cv_auc']:.3f}+/-{m['cv_auc_std']:.3f}  HO-AUC={ho}")
    except Exception as e:
        print(f"  {name}: SKIP ({e})")
results = pd.DataFrame(rows)
results

## 4 — Extraction-time table

Wall-clock time per (dataset, tool): from `TIMINGS` (measured during this session)
with a fallback that parses the tool logs for a `Total elapsed: X min` / `done in X min` line.


In [20]:
import re

def _elapsed_from_log(logpath):
    if not Path(logpath).exists():
        return None
    txt = Path(logpath).read_text(encoding="utf-8", errors="replace")
    hits = re.findall(r"(?:Total elapsed:|done in)\s*([\d.]+)\s*min", txt)
    return float(hits[-1]) if hits else None

def timing_table():
    tools = ["minut", "asari", "ffm", "xcms"]
    data = {}
    for ds in ("covid", "lung", "milk"):
        for t in tools:
            mins = TIMINGS.get((ds, t))
            if mins is None:
                mins = _elapsed_from_log(LOG_DIR / f"{t}_{ds}.log")
            data.setdefault(ds, {})[t] = (round(mins, 2) if mins is not None else None)
    return pd.DataFrame(data).T[tools]

timing_table()


,minut,asari,ffm,xcms
covid,None,None,None,None
lung,None,None,None,None
milk,None,None,None,None


## 5 — Full-cohort reproduction (end-to-end)

`run_all(...)` runs the four extractors and reports the timing table, skipping any output that
already exists. The reference tools are long on full cohorts; MINUT is fast.

In [21]:
def run_all(datasets=("covid", "lung", "milk"), tools=("minut", "ffm", "asari", "xcms")):
    for ds in datasets:
        man = MANIFESTS.get(ds)
        if man is None or not Path(man).exists():
            print(f"[MISS] manifest not found for {ds}: {man}"); continue
        for t in tools:
            out = FEAT_DIR / f"{ds}_{t}.csv"
            if out.exists() and out.stat().st_size > 1024:
                print(f"[SKIP] {ds}/{t} already present"); continue
            if t == "minut":
                extract_minut(ds, man)
            else:
                run_reference_tool(t, ds, man)
    print("\n--- timing ---")
    return timing_table()

# run_all(datasets=("milk",), tools=("minut",))   # small example
# run_all()                                        # everything (LONG)
print("Call run_all(...) for the full reproduction.")


Call run_all(...) for the full reproduction.


## 6 — Case studies (biomarkers)

Downstream biomarker analyses (discriminative-bin selection, annotation, figures) are provided as
separate, self-contained notebooks in the repository (`CaseStudy/`): milk / PABA, lung /
phenylalanylphenylalanine, and COVID-19 / C19-sphingosine & valine. Each starts from the MINUT
feature matrices produced above and applies the selection described in the paper
(Gini importance, and Mann-Whitney + Fisher score for COVID-19).


## 7 — Reproducibility notes

- **Classifier**: `RandomForestClassifier(n_estimators=100, random_state=2024)` — used in section 3
  to score the **reference tools** (asari/FFM/XCMS). MINUT's own accuracy and biomarker analysis
  are in the `CaseStudy/` notebooks (paper's in-fold feature selection), not here.
- **MINUT transform**: implemented **inline** in section 1 (NumPy + optional Numba). Numba only
  accelerates the loop; results are identical to the pure-Python path (max-intensity is exact;
  m/z / RT differ only by float32 rounding, well below 1 ppm).
- **Reference tools**: asari, pyOpenMS FeatureFinderMetabo, XCMS at published defaults (see `scripts/`).
- **Data**: COVID = MTBLS2291, Lung = MTBLS6990 (public); milk = private (TOFOO).
- **Paths**: only `PROJECT_ROOT` and `DATA_DIR` (section 0) are environment-specific.